# 조합F + 5Day_Return — LogisticRegression

## 실험 목적

HF의 `full/index_price_dev.parquet`에서 `코스피 200`만 선택해 5Day_Return 추가 실험을 수행한다.
피처와 미래 5거래일 방향 라벨은 parquet 원시값으로 다시 계산한다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 데이터 | HF 비공개 저장소의 고정 리비전 parquet |
| 홀드아웃 시작 | `20240901` — 이후 행 접근 금지 |
| 외부 검증 | expanding walk-forward 12폴드 |
| 최초 외부 학습 | 750거래일 |
| 외부 검증 | 폴드당 60거래일 |
| 학습·검증 gap | 5거래일 |
| 클래스 가중치 | 외부 폴드마다 내부 60일 검증으로 `None`/`balanced` 선택 |
| 가중치 선정 기준 | Accuracy·Macro F1·하락 Recall 조화평균 |
| 매매 평가 | 상승만 다음 시가 진입, 5슬리브·5거래일 보유, 왕복비용 0.05% |


In [1]:
# 어느 폴더에서 실행해도 프로젝트 모듈을 찾도록 루트를 확인합니다.
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from features.model_dataset import build_model_dataset  # noqa: E402
from models.experiment import evaluate_nested_class_weights  # noqa: E402
from models.notebook_experiment import summarize_notebook_experiment  # noqa: E402
from supply.hf_model_data import load_hf_index_prices  # noqa: E402

COMBINATION = "F"
RETURN_FEATURES = ('five_day_return',)
MODEL_NAME = "LogisticRegression"
CACHE_NAME = "F_five_day_LogisticRegression.json"

# MANIFEST와 parquet를 같은 HF 커밋에서 받고 SHA-256·홀드아웃 경계를 검사합니다.
snapshot = load_hf_index_prices()
dataset = build_model_dataset(
    snapshot.frame,
    COMBINATION,
    return_features=RETURN_FEATURES,
)
print("HF 리비전:", snapshot.repo_sha)
print("지수 parquet SHA-256:", snapshot.file_sha256)
print("학습 가능 기간:", dataset.frame["bas_dd"].min(), "~", dataset.frame["bas_dd"].max())
print("학습 가능 행:", len(dataset.frame))
print("피처:", list(dataset.feature_columns))


C:\Users\Administrator\Alpha_Stack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF 리비전: cf3759afebf73a59c8f6c9aa7265cccb56a38f27
지수 parquet SHA-256: 376c66434db688f42972d2011baee330641421f6b2ca32a8074a96db85ebc13a
학습 가능 기간: 20110127 ~ 20240822
학습 가능 행: 3343
피처: ['sma_gap_20_60', 'sma_gap_5_20', 'rsi_14', 'macd_hist_atr', 'atr_ratio', 'bb_bandwidth', 'hv_regime', 'obv_slope_20', 'five_day_return']


## 클래스 가중치 재튜닝

외부 검증 60일은 가중치 선정에 사용하지 않는다.

In [2]:
# 12개 외부 폴드 각각에서 과거 데이터만으로 가중치를 고르고 OOS를 평가합니다.
nested = evaluate_nested_class_weights(dataset, model_names=(MODEL_NAME,))
result = summarize_notebook_experiment(dataset, nested, MODEL_NAME)

fold_columns = [
    "fold",
    "selected_class_weight",
    "train_size",
    "train_end",
    "valid_start",
    "valid_end",
    "accuracy",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
    "delta_sharpe_net",
    "all_cash",
]
display(result.fold_results.loc[:, fold_columns].round(4))
display(result.weight_counts)


,fold,selected_class_weight,train_size,train_end,valid_start,valid_end,accuracy,macro_f1,down_recall,core_harmonic_mean,delta_sharpe_net,all_cash
0,1,NaN,750,20140207,20140217,20140514,0.3167,0.2362,0.3846,0.3003,-0.0599,False
1,2,balanced,980,20150115,20150123,20150421,0.5167,0.2271,0.0000,0.0000,-3.2595,True
2,3,balanced,1210,20151217,20151228,20160328,0.4667,0.4152,0.1538,0.2715,0.2197,False
3,4,balanced,1439,20161124,20161202,20170228,0.5833,0.2536,0.0000,0.0000,-3.5274,True
4,5,NaN,1669,20171103,20171113,20180207,0.4833,0.2613,0.0000,0.0000,0.6622,False
5,6,NaN,1899,20181016,20181024,20190118,0.3500,0.2556,0.0000,0.0000,-1.0654,False
6,7,balanced,2129,20190920,20190930,20191224,0.3667,0.3200,0.5556,0.3921,1.6404,False
7,8,balanced,2359,20200825,20200902,20201130,0.3500,0.3217,0.8000,0.4158,-0.8586,False
8,9,balanced,2589,20210729,20210806,20211105,0.1333,0.0986,0.0417,0.0720,-0.3248,False
9,10,balanced,2818,20220706,20220714,20221012,0.5500,0.4418,0.6957,0.5435,1.4390,False


,클래스 가중치,선택 폴드 수
0,None,3
1,balanced,9


## OOS 핵심지표와 백테스트 성과

In [3]:
# 전체 720개 OOS 예측을 합쳐 분류 성능과 혼동행렬을 계산합니다.
summary = result.summary
summary_table = pd.Series(
    {
        "전체 OOS 표본": summary["oos_rows"],
        "Accuracy": summary["accuracy"],
        "Macro F1": summary["macro_f1"],
        "하락 Recall": summary["down_recall"],
        "핵심지표 조화평균": summary["core_harmonic_mean"],
        "Balanced Accuracy": summary["balanced_accuracy"],
        "최빈 클래스 기준선": summary["majority_accuracy"],
        "ΔSharpe_net 폴드 중앙값": summary["delta_sharpe_net_median"],
        "전 기간 현금 폴드": summary["all_cash_folds"],
    },
    name="결과",
)
display(summary_table.to_frame().round(4))
display(result.confusion)
display(result.class_report.round(4))

# 뒤의 05.모델비교 노트북은 네 모델의 이 실행 결과만 읽습니다.
payload = {
    "source": {
        "repo_sha": snapshot.repo_sha,
        "index_sha256": snapshot.file_sha256,
        "dev_end": snapshot.dev_end,
    },
    "experiment": {
        "combination": COMBINATION,
        "return_features": list(RETURN_FEATURES),
        "model": MODEL_NAME,
    },
    "summary": summary,
    "weight_counts": result.weight_counts.to_dict(orient="records"),
}
cache_path = project_root / "data" / "raw" / "model_results" / CACHE_NAME
cache_path.parent.mkdir(parents=True, exist_ok=True)
cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("비교용 실행 결과 저장:", cache_path)


,결과
전체 OOS 표본,720.0000
Accuracy,0.4000
Macro F1,0.3764
하락 Recall,0.2809
핵심지표 조화평균,0.3442
Balanced Accuracy,0.3779
최빈 클래스 기준선,0.4181
ΔSharpe_net 폴드 중앙값,0.0799
전 기간 현금 폴드,2.0000


,예측 하락,예측 중립,예측 상승
실제 하락,50,65,63
실제 중립,59,163,79
실제 상승,71,95,75


,precision,recall,f1-score,support
하락,0.2778,0.2809,0.2793,178.0
중립,0.5046,0.5415,0.5224,301.0
상승,0.3456,0.3112,0.3275,241.0
accuracy,0.4000,0.4000,0.4000,0.4
macro avg,0.3760,0.3779,0.3764,720.0
weighted avg,0.3953,0.4000,0.3971,720.0


비교용 실행 결과 저장: C:\Users\Administrator\Alpha_Stack\data\raw\model_results\F_five_day_LogisticRegression.json


In [4]:
# 05-백테스트·성과 형식으로 핵심 결과를 함께 해석합니다.
display(Markdown(f"""
## 결과 해석

- 전체 OOS Accuracy: `{summary['accuracy']:.4f}`
- 전체 OOS Macro F1: `{summary['macro_f1']:.4f}`
- 하락 Recall: `{summary['down_recall']:.4f}`
- 세 핵심지표 조화평균: `{summary['core_harmonic_mean']:.4f}`
- 최빈 클래스 Accuracy 기준선: `{summary['majority_accuracy']:.4f}`
- 비용 차감 ΔSharpe 폴드 중앙값: `{summary['delta_sharpe_net_median']:.4f}`
- 전 기간 현금 보유 폴드: `{summary['all_cash_folds']}`개
- 예측 개수: 하락 `{summary['predicted_down']}` · 중립
  `{summary['predicted_neutral']}` · 상승 `{summary['predicted_up']}`

이 값은 봉인 홀드아웃이 아닌 개발구간의 12폴드 OOS 결과다. 클래스 가중치는 각 폴드의
외부 검증값을 보지 않고, 그보다 앞선 내부 60거래일의 조화평균으로만 선택했다.
"""))



## 결과 해석

- 전체 OOS Accuracy: `0.4000`
- 전체 OOS Macro F1: `0.3764`
- 하락 Recall: `0.2809`
- 세 핵심지표 조화평균: `0.3442`
- 최빈 클래스 Accuracy 기준선: `0.4181`
- 비용 차감 ΔSharpe 폴드 중앙값: `0.0799`
- 전 기간 현금 보유 폴드: `2`개
- 예측 개수: 하락 `180` · 중립
  `323` · 상승 `217`

이 값은 봉인 홀드아웃이 아닌 개발구간의 12폴드 OOS 결과다. 클래스 가중치는 각 폴드의
외부 검증값을 보지 않고, 그보다 앞선 내부 60거래일의 조화평균으로만 선택했다.
